<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Vantage In-database Graph Analysis: Shortest Path & Topology Demo
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>What is Vantage Graph Engine:</b></p>
<img src = 'image/simple_network.png' width = '600'>
Vantage Graph Engine is a distributed designed specifically to handle large-scale graph data—data made of nodes and relationships.
Vantage Graph Engine combines:
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>A flexible graph data model</li>
    <li>Designed with Teradata MPP architecture</li>
    <li>Written in dynamic (recurvise) queries and stored procedure</li>
    <li>Utilized volatile table for transactional data</li>
</ul>
Together, these components allow fast graph traversal, querying, analytics, and real‑time processing.

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Why use Vantage Graph Engine:</b></p>
Graph Engines excel where data is heavily interconnected and must be queried quickly. Examples include:

<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>Finding shortest paths</li>
    <li>Exploring neighborhood relationships</li>
    <li>Running graph neural network workloads</li>
    <li>Performing multi-hop queries</li>
    <li>Real-time analytics</li>
</ul>
<hr>

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Common Use Cases for Graph Engine:</b></p>
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li><b>Knowledge Graphs & Semantic Networks:</b>Type-safe modeling + scalable traversal; Semantic search & question answering</li>
    <li><b>Recommendation Systems:</b> Personal product and content recommendations, Behavioral similarity scoring</li>
    <li><b>High-Throughput Graph Analytics:</b>Dependency analysis, Shortest paths</li>
    <li>etc.</li>
</ul>



<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Data Structure:</b></p>
<table style = 'width:100%;table-layout:fixed;font-family:Arial;color:#00233C'>
    <tr><td style = 'vertical-align:top' width = '50%'>
        <img src = 'image/simple_network.png' width = '600'></td>
        <td><img src = 'image/node_edge_tables.png' width = '600'></td>
    </tr>
</table>

<hr>

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Demo Overview</b></p>
<p style = 'font-size:16px;font-family:Arial;color:#00233C'>Vantage provides Stored Procedures that simulate the Graph functions executed within the platform and leverage the Teradata MPP architecture. This notebook demonstrates the following functions:</p>

<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li><b>td_shortest_path</b> determines the shortest path between two nodes is the path whose total weights (or total paths) is minimal. Dijkstra’s Algorithm is used in the current logic</li>
    <li><b>td_topology</b> identifies the direct and indirect connections from a source node. It can be used to identify all paths(P) or nodes(N) that are connected to the source.</li>
</ul>

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Demonstration Data</b></p>
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>Very small set of weighted network data. This dataset can be easily visualized.</li>
    <li>california Roadnet: This is a big dataset without weight and it does not come with the label for the intersaction</li>
    <li>IMDB (old movies): Dataset from IMDB with the movies and people (writer, director, actor, etc...)</li>
    
</ul>

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Load Library</b></p>

In [ ]:
# Teradata python package
from teradataml import *
from TeradataGE import td_graph_function, configure

import pandas as pd
import json
import getpass
import numpy as np

# for local display only
import matplotlib.pyplot as plt
from matplotlib import cm, colors


In [ ]:
configure.graph_install_location = "GraphLib"

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Connect to Teradata Database</b></p>

In [ ]:
hostname = "xx.xx.xx.xx"
user = input(prompt=f"Username for database:")
password = getpass.getpass(prompt=f"Database password for {user}:")
databasename = "graphdb"
logmech = "TD2"

eng = create_context(host = hostname, username = user, password = password, database= databasename, logmech = logmech)
print(eng)

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Demo Dataset #1: Simplified weighted network</b></p>
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>Very small dataset</li>
    <li>Directed</li>
    <li>Weighted</li>
</ul>

# TD Traversal Analysis:

In [ ]:
# Define node and edge metadata
graph_obj = td_graph_function.td_graph_object(database_name = 'graphdb',
                                              edge_table_name = 'graph_edge',
                                              edge_from_node_column_name = 'from_node',
                                              edge_to_node_column_name = 'to_node',
                                              edge_weight_column_name = 'edge_weight',
                                              node_table_name = 'graph_node',
                                              node_id_column_name = 'id',
                                              node_label_column_name = 'node_name'
                                              )
print(graph_obj.node_info())
print(graph_obj.edge_info())


In [ ]:
graph_obj.set_node_id_from_label_source('1')

In [ ]:
graph_obj.set_node_id_from_id([1,8])
#graph_obj.set_node_id_from_id()

In [ ]:
# Directed Path
all_path_from_1node = graph_obj.td_topology(output_table = 'example_topology_path_vt')
all_path_from_1node.sort("path_level")

In [ ]:
# Decode  the topology
path_label_df, fig = graph_obj.td_graph_path_decode(edge_attributes = 'edge_weight', include_plot=True)

In [ ]:
# Display in table form
path_label_df

In [ ]:
# Display in diagram
fig.show()

# TD Shortest Path:

In [ ]:
# shortest Path
shortest_path_df = graph_obj.td_shortest_path(source = 1, target = 5, output_table = 'example_shortest_path_vt')
shortest_path_df

In [ ]:
# Display  the topology
path_label_df, fig = graph_obj.td_graph_path_decode(edge_attributes = 'edge_weight', include_plot=True)

In [ ]:
fig

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Demo Dataset #2: Compact Unweigthed Network: California Road Network</b></p>
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>Large dataset</li>
    <li>Directed</li>
    <li>Un-weighted</li>
</ul>

In [ ]:
# Define edge metadata
graph_obj = td_graph_function.td_graph_object(edge_table_name = 'roadnet_ca_edge',
                                              edge_from_node_column_name = 'from_id',
                                              edge_to_node_column_name = 'to_id',
                                              node_table_name = 'roadnet_ca_node',
                                              node_id_column_name = 'node_id',
                                              node_label_column_name = 'node_label'
                                             )
print(graph_obj.edge_info())


In [ ]:
# Display all possible pathway
from_id = 0
graph_obj.set_node_id_from_id(from_id)
all_path_from_1node = graph_obj.td_topology(max_path_length = 20, return_data = 'N')
all_path_from_1node

In [ ]:
# Display all nodes that has between thru
graph_obj.set_node_id_from_id(from_id)
all_node_from_1node = graph_obj.td_topology(max_path_length = 20, return_data = 'N')

print(f"# of possible from {from_id} = {all_node_from_1node.shape[0]}")
print(all_node_from_1node)

<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Demo Dataset #3: Network with Relationship: IMDB Graph topology</b></p>
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>Large dataset</li>
    <li>Directed</li>
    <li>Relationship</li>
</ul>

In [ ]:
# Define both edge and node metadata
graph_obj = td_graph_function.td_graph_object(edge_table_name = 'imdb_graph_edge',
                                              edge_from_node_column_name = 'from_id',
                                              edge_to_node_column_name = 'to_id',
                                              node_table_name = 'imdb_graph_node',
                                              node_id_column_name = 'node_id',
                                              node_label_column_name = 'primaryname'
                                             )


In [ ]:
# Display the defined information for edge
print(graph_obj.edge_info())

In [ ]:
# Display the defined information for node
print(graph_obj.node_info())

In [ ]:
# Setup the source id from node label
graph_obj.set_node_id_from_label_source("The Last of the Mafia")

In [ ]:
# Idenfify 5 level of relationship from source
path_df = graph_obj.td_topology(max_path_length = 5, output_table ='imdb_path')
path_df

In [ ]:
# Converting the path from id to human readiable results
path_label_df, fig = graph_obj.td_graph_path_decode('imdb_path', edge_attributes=['attr1', 'attr2'], include_plot=True)
fig

In [ ]:
source_movie = "Midnight Madness"
target_movie = "The Last of the Mafia"
#graph_obj.set_node_id_from_label_source(source_movie)
#graph_obj.set_node_id_from_label_target(target_movie)


shortest_path_df = graph_obj.td_shortest_path(
    source = source_movie,
    target = target_movie, 
    max_path_length = 100, output_table ='movies_relationship', temp_output_table=True)
shortest_path_df

In [ ]:
# Converting the path from id to human readiable results
path_label_df, fig = graph_obj.td_graph_path_decode('movies_relationship', edge_attributes=['attr1', 'attr2'], include_plot=True)
fig

# Disconnect from database

In [ ]:
remove_context()